In [15]:
# importing required libraries
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import requests
import ollama

In [12]:
# loading the environment variables
load_dotenv(override=True) # load the environment variables from the .env file
openai_api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
claude_api_key = os.getenv('CLAUDE_API_KEY')


# validate whether the API keys are loaded correctly
# OpenAI
if not openai_api_key:
    print("OPENAI_API_KEY is missing")
elif not openai_api_key.startswith("sk-proj-"):
    print("OPENAI_API_KEY is set, but does not start with sk-proj-")
else:
    print(f"OPENAI_API_KEY loaded, starts with {openai_api_key[:8]}")

# Gemini (this course also uses GOOGLE_API_KEY)
if not gemini_api_key:
    print("GEMINI_API_KEY is missing — check the name in .env")
elif not gemini_api_key.startswith(("AIz", "AQ.")):
    print("GEMINI_API_KEY is set, but does not start with AIz or AQ.")
else:
    print(f"GEMINI_API_KEY loaded, starts with {gemini_api_key[:4]}")

# Claude
if not claude_api_key:
    print("CLAUDE_API_KEY is missing — check the name in .env")
elif not claude_api_key.startswith("sk-ant-"):
    print("CLAUDE_API_KEY is set, but does not start with sk-ant-")
else:
    print(f"CLAUDE_API_KEY loaded, starts with {claude_api_key[:8]}")

OPENAI_API_KEY loaded, starts with sk-proj-
GEMINI_API_KEY loaded, starts with AIza
CLAUDE_API_KEY loaded, starts with sk-ant-a


In [11]:
class PromptBuilder:
    """Fetched page: url + text. Builds chat messages for summarization."""

    SYSTEM_PROMPT = "summarize the content of a website"
    USER_PROMPT_PREFIX = "provide me highlights of the news in bullet points from the content of website:"
    
    def __init__(self, url: str):
        self.url = url
        self.html_page = fetch_website_contents(url)
        
    def messages(self) -> list[dict]:
        return [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": self.USER_PROMPT_PREFIX + self.html_page},
        ]

In [9]:
class LLMModel:
    """Any OpenAI-compatible endpoint (Gemini, Ollama, OpenAI)."""

    def __init__(self, name:str, client: OpenAI, model: str):
        self.name = name
        self.client = client
        self.model = model
        
    def complete(self, messages: list[dict]) -> str:
        # send the messages to the model and return the response
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        return response.choices[0].message.content

In [10]:
class WebsiteSummarizer:
    """Orchestrates: fetch URL → messages → model → markdown."""

    def __init__(self, llm_model: LLMModel):
        self.llm_model = llm_model

    def summarize(self, url: str) -> str:
        promptBuilder = PromptBuilder(url)
        payload = promptBuilder.messages() # build the messages for the model
        return self.llm_model.complete(payload) # return the model's response

    def display(self, url: str) -> None:
        print(f"{self.llm_model.name} ({self.llm_model.model})\n{url}\n")
        display(Markdown(self.summarize(url))) # display the model's response in markdown format

## **Gemini Chat Completions API**

In [ ]:
gemini_model = LLMModel(
    name = 'Gemini',
    client = OpenAI(
        base_url = "https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key = gemini_api_key,
    ),
    model="gemini-3.1-flash-lite",
)

gemini_summarizer = WebsiteSummarizer(gemini_model)
gemini_summarizer.display("https://www.cnn.com/2026/08/30/world/live-news/nepal-china-flood")


Gemini (gemini-3.1-flash-lite)
https://www.cnn.com/2026/08/30/world/live-news/nepal-china-flood



Based on the title and context provided for the CNN report, here are the key highlights regarding the flooding in China and Nepal:

*   **Rising Death Toll:** The number of confirmed fatalities from the flooding in China and Nepal has continued to increase as the situation develops.
*   **Ongoing Search and Rescue:** Emergency teams and rescuers are actively searching for missing individuals, specifically focusing on workers who were caught in the path of the floods.
*   **Regional Impact:** The disaster is affecting both China and Nepal, with significant damage and hazardous conditions reported across the impacted areas.
*   **Critical Conditions:** Heavy rainfall and subsequent flooding have triggered search operations in difficult terrain, complicating efforts to reach those still missing.

*(Note: The text provided in your prompt was primarily the navigation and ad-feedback interface of the CNN website; the summary above is based on the subject matter defined by the page title.)*

## **OLLAMA Chat Completions API**

Browse models: https://ollama.com/library/

In [ ]:
class OllamaModel(LLMModel):
    """Local Ollama: ping, pull, list, then chat via the OpenAI-compatible API."""

    BASE_URL = "http://localhost:11434"

    def __init__(self, model: str = "llama3.2:1b", name: str = "Ollama"):
        client = OpenAI(base_url=f"{self.BASE_URL}/v1")
        super().__init__(name=name, client=client, model=model)

    def is_running(self) -> bool:
        try:
            response = requests.get(self.BASE_URL, timeout=3)
            return response.ok and "running" in response.text.lower()
        except requests.RequestException:
            return False

    def ensure_running(self) -> None:
        if not self.is_running():
            raise RuntimeError(
                f"Ollama is not running at {self.BASE_URL}. Start it, then retry."
            )
        print(requests.get(self.BASE_URL).text)

    def pull(self) -> None:
        self.ensure_running()
        ollama.pull(self.model)

    def list_models(self) -> None:
        print(ollama.list())
        for entry in ollama.list().models:
            print(entry.model)

In [22]:
llama_model = OllamaModel(model='llama3.2:1b')
llama_model.pull()
llama_model.list_models()

Ollama is running
Ollama is running
models=[Model(model='llama3.2:1b', modified_at=datetime.datetime(2026, 8, 30, 22, 5, 57, 747837, tzinfo=TzInfo(-04:00)), digest='baf6a787fdffd633537aa2eb51cfd54cb93ff08e28040095462bb63daf552878', size=1321098329, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='1.2B', quantization_level='Q8_0'))]
llama3.2:1b


In [23]:
# llama_model = LLMModel(
#     name="Ollama",
#     client=OpenAI(base_url = 'http://localhost:11434/' + "v1"),
#     model="llama3.2:1b",
# )

llama_summarizer = WebsiteSummarizer(llama_model)
llama_summarizer.display("https://www.bbc.com/news")

Ollama (llama3.2:1b)
https://www.bbc.com/news



Here are the highlights of the news in bullet points:

* **US strikes Iranian launchers on Larak Island**: The US has launched a surprise attack on Iranian launchers on Larak Island, killing some civilians, according to Iran. This is the US's first known attack on Iran since late July.
* **Nepal struggles to free trapped workers**: Nepal is intensifying efforts to free hundreds of workers believed trapped in hydropower tunnels, with India and China deploying experts to assist.
* **US and India deploy experts**: The US is sending experts to assist in the operation, while India has deployed its own technical team to help with the rescue efforts.
* **China sends aid**: China is sending aid to Nepal, including rescue equipment and personnel, to assist in the rescue efforts.
* **Iran denies involvement**: Iran has denied being involved in the attack on Larak Island, although it has vowed to protect its interests in the region.
* **US strikes come amid tensions**: The US has made multiple attacks on Iranian military sites in recent weeks, amid high levels of tension in the region.

Fun fact demo

In [24]:
display(Markdown(llama_model.complete([{"role": "user", "content": "Tell me a fun fact"}])))

A fun fact is that there is a species of jellyfish that is immortal. The Turritopsis dohrnii, also known as the "immortal jellyfish," is a type of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making the Turritopsis dohrnii theoretically immortal.